GPU Enabled

In [1]:
!nvidia-smi

Mon Aug 17 12:54:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Full-corpus run: gemma4:12b, one state at a time

Colab caps sessions at 24h. One state's ±1yr AfD-entry window fits in that. The full corpus does not.

`score_with_model.py --full-corpus` checkpoints every row and resumes on reconnect.

Model: `gemma4:12b`, not `gemma4:e2b` (the small on-device variant).

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ['DATA_ROOT'] = "/content/drive/MyDrive/my_projects/M.A. Parliament/Code and Data/data"

Mounted at /content/drive


In [ ]:
# repo is public, plain clone, no auth needed
import os

if not os.path.isdir('/content/Incivility-in-Plenary-de'):
    !git clone https://github.com/AnLeWe/Incivility-in-Plenary-de.git /content/Incivility-in-Plenary-de
else:
    !cd /content/Incivility-in-Plenary-de && git pull

%cd /content/Incivility-in-Plenary-de
!pip install -q -r requirements.txt

In [ ]:
# state for this run. rows in each state's ±1yr AfD-entry window: by=72089, th=69492, sn=67114
STATE = "sn"

Pull the model in the terminal (Runtime, Open terminal), not inline. A progress bar does not render well in a notebook cell.

```
ollama pull gemma4:12b
```

In [4]:
!sudo apt-get update -y
!sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!OLLAMA_NUM_PARALLEL=8 nohup ollama serve > /content/ollama_serve.log 2>&1 &
!sleep 5 && ollama pull gemma4:12b
!ollama list


Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [107 kB]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,578 B]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,908 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]          
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,164 kB]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
G

## Concurrency benchmark (this GPU specifically)

`OLLAMA_NUM_PARALLEL=8` is set in the install cell above. If that cell already ran once before this note existed, restart the Colab runtime and re-run from the top.

Times real rows for the current `STATE` at concurrency 1, 2, 4, 8, 16 with `gemma4:12b`. The Mac benchmark plateaued at concurrency 2. This GPU may not. Measure here, then set `CONCURRENCY` below to whatever wins.

```python
import sys, time
sys.path.insert(0, '/content/Incivility-in-Plenary-de/measurement')
from concurrent.futures import ThreadPoolExecutor

import ollama
from impoliteness_lib import build_classification_pool, build_prompt, PROMPT_VERSION

pool = build_classification_pool(os.environ['DATA_ROOT'], verbose=False)
sample_texts = (
    pool.dedup[pool.dedup['state'] == STATE]['text_to_classify']
    .dropna().astype(str).head(16).tolist()
)

def call_one(text):
    t0 = time.time()
    ollama.chat(
        model='gemma4:12b',
        messages=build_prompt(text, prompt_version=PROMPT_VERSION),
        format='json', think=False,
        options={'temperature': 0, 'seed': 20260723, 'num_ctx': 40960},
    )
    return time.time() - t0

for concurrency in (1, 2, 4, 8, 16):
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=concurrency) as ex:
        list(ex.map(call_one, sample_texts))
    wall = time.time() - t0
    print(f"concurrency={concurrency}: {wall/len(sample_texts):.2f}s/item, {len(sample_texts)/wall:.2f} items/s")
```

In [ ]:
# set from the benchmark above, whatever concurrency actually won
CONCURRENCY = 4

In [ ]:
# baseline, window, flag, or window_flag
VARIANT = "baseline"

## Run

Scores `STATE` with `VARIANT`, checkpointed, resumes on reconnect. The completed Bavaria four-variant run is archived at `run_history/colab_gemma_2026-08-17_gemma4-12b_by_4variants.ipynb`.

In [ ]:
%cd /content/Incivility-in-Plenary-de/measurement
!python score_with_model.py --model gemma4:12b --variant {VARIANT} --state {STATE} --platform-label colab_a100 --concurrency {CONCURRENCY}

## With Transformers

Local inference on GPU. Model page: https://huggingface.co/google/gemma-4-12B-it

In [ ]:
!pip install -U transformers

In [ ]:
# Load model directly
from transformers import AutoProcessor, AutoModelForMultimodalLM

processor = AutoProcessor.from_pretrained("google/gemma-4-12B-it")
model = AutoModelForMultimodalLM.from_pretrained("google/gemma-4-12B-it", device_map="auto")